In [ ]:
import os
import warnings
from glob import glob
from itertools import combinations_with_replacement

import numpy as np
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()

from pinn.io import load_qm9, sparse_batch

try:
    from pinn.models.potential import potential_model
except ImportError:
    from pinn.models import potential_model

os.environ["CUDA_VISIBLE_DEVICES"] = ""

warnings.filterwarnings("ignore", "Converting sparse IndexedSlices")
QM9_DIR = "../dsgdb9nsd"     # xyz files of atoms
MODEL_DIR = "./BPNN_QM9_U0"

ATOM_TYPES = [1, 6, 7, 8, 9]          # H, C, N, O, F

BATCH_SIZE = 32
MAX_STEPS = 20000
EVAL_STEPS = 200

USE_ANGULAR_G4 = True

filelist = sorted(glob(os.path.join(QM9_DIR, "*.xyz")))

if len(filelist) == 0:
    raise RuntimeError(
        f"No QM9 .xyz files found in {QM9_DIR}. "
        "Please check QM9_DIR."
    )

print(f"Found {len(filelist)} QM9 files.")

ELEMENT_TO_Z = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "F": 9,
}

def read_qm9_energy_and_counts(filename):
    """
    Read one QM9 .xyz file.

    Returns:
        U0 energy and atom-count dictionary.
    """

    with open(filename, "r") as f:
        lines = f.readlines()

    natoms = int(lines[0].strip())

    fields = lines[1].split()

    if len(fields) >= 17:
        U0 = float(fields[12])
    else:
        U0 = float(fields[11])

    counts = {z: 0 for z in ATOM_TYPES}

    for line in lines[2:2 + natoms]:
        symbol = line.split()[0]
        z = ELEMENT_TO_Z[symbol]
        counts[z] += 1

    return U0, counts


def estimate_atomic_dress_from_qm9(filelist, atom_types, max_files=20000):
    """
    Fit:
        E_molecule ≈ sum_Z n_Z * e_dress[Z]

    This removes the large absolute atomic-energy offset from QM9.
    """

    X = []
    y = []

    for filename in filelist[:max_files]:
        energy, counts = read_qm9_energy_and_counts(filename)
        X.append([counts[z] for z in atom_types])
        y.append(energy)

    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)

    e_dress = {
        int(z): float(e)
        for z, e in zip(atom_types, coeffs)
    }

    return e_dress


e_dress = estimate_atomic_dress_from_qm9(
    filelist=filelist,
    atom_types=ATOM_TYPES,
    max_files=min(20000, len(filelist)),
)

print("\nEstimated atomic dress:")
for z, e in e_dress.items():
    print(f"  Z={z}: {e:.8f}")


# Behler-Parrinello symmetry functions

def make_bpnn_sf_spec(atom_types, use_angular=True):
    """
    Build a PiNN BPNN symmetry-function specification.

    Important fix:
        For G2, Rs and eta must have compatible lengths.

    Therefore each G2 entry has:
        len(Rs) == len(eta)
    """

    sf_spec = []

    # Radial G2 functions
    radial_Rs = [0.0, 0.8, 1.6, 2.4, 3.2, 4.0]
    radial_etas = [0.05, 0.2, 0.8]

    for i in atom_types:
        for j in atom_types:
            for eta in radial_etas:
                sf_spec.append({
                    "type": "G2",
                    "i": int(i),
                    "j": int(j),
                    "Rs": radial_Rs,
                    "eta": [eta] * len(radial_Rs),
                })

    # Angular G4 functions
    if use_angular:
        angular_etas = [0.005, 0.02, 0.08]
        angular_zetas = [1.0, 4.0]
        angular_lambdas = [-1.0, 1.0]

        for i in atom_types:
            for j in atom_types:
                for eta in angular_etas:
                    for zeta in angular_zetas:
                        for lambd in angular_lambdas:
                            sf_spec.append({
                                "type": "G4",
                                "i": int(i),
                                "j": int(j),
                                "eta": [eta],
                                "zeta": [zeta],
                                "lambd": [lambd],
                            })

    return sf_spec


sf_spec = make_bpnn_sf_spec(
    ATOM_TYPES,
    use_angular=USE_ANGULAR_G4,
)

print(f"\nNumber of symmetry-function blocks: {len(sf_spec)}")

# qm9 dataset

def make_dataset():
    """
    Load QM9 and map U0 -> e_data.

    Important fix:
        use 'splits', not 'split'.
    """

    return load_qm9(
        filelist,
        splits={"train": 8, "test": 2},
        label_map={"e_data": "U0"},
    )


def train_input_fn():
    return (
        make_dataset()["train"]
        .shuffle(5000)
        .repeat()
        .apply(sparse_batch(BATCH_SIZE))
    )


def test_input_fn():
    return (
        make_dataset()["test"]
        .repeat()
        .apply(sparse_batch(BATCH_SIZE))
    )

# PiNN BPNN model

params = {
    "model_dir": MODEL_DIR,

    "network": {
        "name": "BPNN",
        "params": {
            "rc": 5.0,
            "sf_spec": sf_spec,

            "nn_spec": {
                1: [32, 32],     # H
                6: [64, 64],     # C
                7: [64, 64],     # N
                8: [64, 64],     # O
                9: [64, 64],     # F
            },

            "act": "tanh",
            "cutoff_type": "f1",

            "preprocess": False,
            "use_jacobian": False,
            "fp_scale": False,
        },
    },

    "model": {
        "name": "potential_model",
        "params": {
            "use_force": False,
            "use_stress": False,

            "e_dress": e_dress,

            "e_loss_multiplier": 1.0,
            "log_e_per_atom": True,
        },
    },

    "optimizer": {
        "class_name": "Adam",
        "config": {
            "learning_rate": 1e-3,
        },
    },
}

model = potential_model(params)

train_spec = tf.estimator.TrainSpec(
    input_fn=train_input_fn,
    max_steps=MAX_STEPS,
)

eval_spec = tf.estimator.EvalSpec(
    input_fn=test_input_fn,
    steps=EVAL_STEPS,
)

tf.estimator.train_and_evaluate(
    model,
    train_spec,
    eval_spec,
)

eval_result = model.evaluate(
    input_fn=test_input_fn,
    steps=EVAL_STEPS,
)

print("\nFinal evaluation:")
for key, value in eval_result.items():
    print(f"{key}: {value}")

Found 133885 QM9 files.

Estimated atomic dress:
  Z=1: -0.60397747
  Z=6: -38.07294388
  Z=7: -54.75139113
  Z=8: -75.22570431
  Z=9: -99.85409350

Number of symmetry-function blocks: 375
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': './BPNN_QM9_U0', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluste

2026-05-13 20:42:06.723165: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


61472 trainable vaiables, training with float32 precision.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Done calling model_fn.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Create CheckpointSaverHook.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:Graph was finalized.


2026-05-13 20:42:21.513245: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled


INFO:tensorflow:Running local_init_op.
INFO:tensorflow:Done running local_init_op.
INFO:tensorflow:Calling checkpoint listeners before saving checkpoint 0...
INFO:tensorflow:Saving checkpoints for 0 into ./BPNN_QM9_U0/model.ckpt.
INFO:tensorflow:Calling checkpoint listeners after saving checkpoint 0...
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
Instructions for updating:
Use tf.keras instead.
INFO:tensorflow:loss = 135.74469, step = 1
INFO:tensorflow:global_step/sec: 19.3553
INFO:tensorflow:loss = 0.26335406, step = 101 (5.167 sec)
INFO:tensorflow:global_step/sec: 41.7198
INFO:tensorflow:loss = 0.060336307, step = 201 (2.397 sec)
INFO:tensorflow:global_step/sec: 40.697
INFO:tensorflow:loss = 0.06300274, step = 301 (2.457 sec)
INFO:tensorflow:global_step/sec: 40.2275
INFO:tensorflow:loss = 0.050486576, step = 401 (2.486 sec)
INFO:tensorflow:global_step/sec: 39.6879
INFO:tensorflow:loss = 0.0476109, step = 501 (2.520 sec)
INFO:tensor

In [ ]:
import os
import csv
import random
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

from ase import Atoms
from ase.data import covalent_radii

from pinn.calculator import PiNN_calc

try:
    from pinn.models.potential import potential_model
except ImportError:
    from pinn.models import potential_model

QM9_DIR = "../dsgdb9nsd"
MODEL_DIR = "./BPNN_QM9_U0"
OUT_DIR = "./BPNN_QM9_visualization"

ATOM_TYPES = [1, 6, 7, 8, 9]           # H, C, N, O, F

N_SAMPLE = 1000
RANDOM_SEED = 123

USE_ANGULAR_G4 = True

os.makedirs(OUT_DIR, exist_ok=True)

filelist = sorted(glob(os.path.join(QM9_DIR, "*.xyz")))

if len(filelist) == 0:
    raise RuntimeError(f"No QM9 .xyz files found in {QM9_DIR}")

print(f"Found {len(filelist)} QM9 files.")

ELEMENT_TO_Z = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "F": 9,
}

def qm9_float(x):
    """
    Convert QM9 numeric strings to Python float.

    Handles:
        2.1997*^-6  -> 2.1997e-6
        1.23D-04    -> 1.23e-04
    """
    return float(
        x.replace("*^", "e")
         .replace("D", "e")
         .replace("d", "e")
    )


def read_qm9_xyz(filename):
    """
    Read QM9 xyz file.

    Returns:
        atoms: ASE Atoms object
        U0: QM9 U0 target energy
    """

    with open(filename, "r") as f:
        lines = f.readlines()

    natoms = int(lines[0].strip())
    fields = lines[1].split()

    if len(fields) >= 17:
        U0 = qm9_float(fields[12])
    else:
        U0 = qm9_float(fields[11])

    symbols = []
    positions = []

    for line in lines[2:2 + natoms]:
        parts = line.split()

        symbols.append(parts[0])

        positions.append([
            qm9_float(parts[1]),
            qm9_float(parts[2]),
            qm9_float(parts[3]),
        ])

    atoms = Atoms(
        symbols=symbols,
        positions=np.asarray(positions, dtype=float)
    )

    return atoms, U0


def read_qm9_energy_and_counts(filename):
    atoms, U0 = read_qm9_xyz(filename)
    counts = {z: 0 for z in ATOM_TYPES}

    for z in atoms.get_atomic_numbers():
        counts[int(z)] += 1

    return U0, counts


def estimate_atomic_dress_from_qm9(filelist, atom_types, max_files=20000):
    """
    Must match the training script.
    """

    X = []
    y = []

    for filename in filelist[:max_files]:
        energy, counts = read_qm9_energy_and_counts(filename)
        X.append([counts[z] for z in atom_types])
        y.append(energy)

    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)

    e_dress = {
        int(z): float(e)
        for z, e in zip(atom_types, coeffs)
    }

    return e_dress


def make_bpnn_sf_spec(atom_types, use_angular=True):
    """
    Must match the training script.
    """

    sf_spec = []

    radial_Rs = [0.0, 0.8, 1.6, 2.4, 3.2, 4.0]
    radial_etas = [0.05, 0.2, 0.8]

    for i in atom_types:
        for j in atom_types:
            for eta in radial_etas:
                sf_spec.append({
                    "type": "G2",
                    "i": int(i),
                    "j": int(j),
                    "Rs": radial_Rs,
                    "eta": [eta] * len(radial_Rs),
                })

    if use_angular:
        angular_etas = [0.005, 0.02, 0.08]
        angular_zetas = [1.0, 4.0]
        angular_lambdas = [-1.0, 1.0]

        for i in atom_types:
            for j in atom_types:
                for eta in angular_etas:
                    for zeta in angular_zetas:
                        for lambd in angular_lambdas:
                            sf_spec.append({
                                "type": "G4",
                                "i": int(i),
                                "j": int(j),
                                "eta": [eta],
                                "zeta": [zeta],
                                "lambd": [lambd],
                            })

    return sf_spec


e_dress = estimate_atomic_dress_from_qm9(
    filelist=filelist,
    atom_types=ATOM_TYPES,
    max_files=min(20000, len(filelist)),
)

sf_spec = make_bpnn_sf_spec(
    ATOM_TYPES,
    use_angular=USE_ANGULAR_G4,
)

params = {
    "model_dir": MODEL_DIR,

    "network": {
        "name": "BPNN",
        "params": {
            "rc": 5.0,
            "sf_spec": sf_spec,
            "nn_spec": {
                1: [32, 32],
                6: [64, 64],
                7: [64, 64],
                8: [64, 64],
                9: [64, 64],
            },
            "act": "tanh",
            "cutoff_type": "f1",
            "preprocess": False,
            "use_jacobian": False,
            "fp_scale": False,
        },
    },

    "model": {
        "name": "potential_model",
        "params": {
            "use_force": False,
            "use_stress": False,
            "e_dress": e_dress,
            "e_loss_multiplier": 1.0,
            "log_e_per_atom": True,
        },
    },

    "optimizer": {
        "class_name": "Adam",
        "config": {
            "learning_rate": 1e-3,
        },
    },
}

model = potential_model(params)
calc = PiNN_calc(model)

rng = random.Random(RANDOM_SEED)
sample_files = rng.sample(filelist, min(N_SAMPLE, len(filelist)))

rows = []

for n, filename in enumerate(sample_files, start=1):
    atoms, e_true = read_qm9_xyz(filename)

    atoms.set_calculator(calc)
    e_pred = atoms.get_potential_energy()

    natoms = len(atoms)
    error = e_pred - e_true

    rows.append({
        "filename": os.path.basename(filename),
        "formula": atoms.get_chemical_formula(),
        "natoms": natoms,
        "e_true": e_true,
        "e_pred": e_pred,
        "error": error,
        "abs_error": abs(error),
        "error_per_atom": error / natoms,
        "abs_error_per_atom": abs(error) / natoms,
    })

    if n % 100 == 0:
        print(f"Predicted {n}/{len(sample_files)} molecules")


csv_path = os.path.join(OUT_DIR, "qm9_bpnn_predictions.csv")

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved prediction table: {csv_path}")

e_true = np.array([r["e_true"] for r in rows])
e_pred = np.array([r["e_pred"] for r in rows])
err = e_pred - e_true
abs_err = np.abs(err)
natoms = np.array([r["natoms"] for r in rows])
err_per_atom = err / natoms
abs_err_per_atom = abs_err / natoms

mae = np.mean(abs_err)
rmse = np.sqrt(np.mean(err ** 2))
mae_atom = np.mean(abs_err_per_atom)
rmse_atom = np.sqrt(np.mean(err_per_atom ** 2))

print("\nEnergy statistics:")
print(f"  MAE:          {mae:.8f}")
print(f"  RMSE:         {rmse:.8f}")
print(f"  MAE / atom:   {mae_atom:.8f}")
print(f"  RMSE / atom:  {rmse_atom:.8f}")

plt.figure(figsize=(6, 6))
plt.scatter(e_true, e_pred, s=12, alpha=0.6)

lo = min(e_true.min(), e_pred.min())
hi = max(e_true.max(), e_pred.max())

plt.plot([lo, hi], [lo, hi], "--", linewidth=1)

plt.xlabel("QM9 U0 target energy")
plt.ylabel("BPNN predicted energy")
plt.title("BPNN prediction vs QM9 target")
plt.tight_layout()

path = os.path.join(OUT_DIR, "predicted_vs_target.png")
plt.savefig(path, dpi=300)
plt.close()

print(f"Saved: {path}")

plt.figure(figsize=(7, 5))
plt.hist(err, bins=50, alpha=0.8)
plt.axvline(0.0, linestyle="--", linewidth=1)

plt.xlabel("Prediction error: E_pred - E_true")
plt.ylabel("Count")
plt.title("Energy residual distribution")
plt.tight_layout()

path = os.path.join(OUT_DIR, "residual_histogram.png")
plt.savefig(path, dpi=300)
plt.close()

print(f"Saved: {path}")

plt.figure(figsize=(7, 5))
plt.scatter(natoms, abs_err_per_atom, s=12, alpha=0.6)

plt.xlabel("Number of atoms")
plt.ylabel("Absolute error per atom")
plt.title("Size dependence of BPNN error")
plt.tight_layout()

path = os.path.join(OUT_DIR, "abs_error_per_atom_vs_natoms.png")
plt.savefig(path, dpi=300)
plt.close()

print(f"Saved: {path}")

plt.figure(figsize=(7, 5))
plt.scatter(e_true, err, s=12, alpha=0.6)
plt.axhline(0.0, linestyle="--", linewidth=1)

plt.xlabel("QM9 U0 target energy")
plt.ylabel("Prediction error: E_pred - E_true")
plt.title("Residuals vs target energy")
plt.tight_layout()

path = os.path.join(OUT_DIR, "residual_vs_target.png")
plt.savefig(path, dpi=300)
plt.close()

print(f"Saved: {path}")

def find_likely_bond(atoms):
    """
    Find one likely covalent bond using covalent radii.
    """

    numbers = atoms.get_atomic_numbers()
    positions = atoms.get_positions()

    best_pair = None
    best_dist = None

    for i in range(len(atoms)):
        for j in range(i + 1, len(atoms)):
            rij = np.linalg.norm(positions[j] - positions[i])
            cutoff = 1.25 * (covalent_radii[numbers[i]] + covalent_radii[numbers[j]])

            if rij < cutoff:
                if best_dist is None or rij < best_dist:
                    best_pair = (i, j)
                    best_dist = rij

    return best_pair, best_dist


def make_bond_scan(atoms, calc, n_points=60):
    """
    Stretch one detected bond and compute predicted energy.
    """

    pair, r0 = find_likely_bond(atoms)

    if pair is None:
        return None

    i, j = pair

    positions0 = atoms.get_positions()
    direction = positions0[j] - positions0[i]
    direction /= np.linalg.norm(direction)

    distances = np.linspace(max(0.6, r0 - 0.4), r0 + 1.2, n_points)

    energies = []

    for r in distances:
        atoms_scan = atoms.copy()
        positions = positions0.copy()
        positions[j] = positions[i] + r * direction
        atoms_scan.set_positions(positions)
        atoms_scan.set_calculator(calc)
        energies.append(atoms_scan.get_potential_energy())

    return pair, r0, distances, np.asarray(energies)


atoms0, _ = read_qm9_xyz(sample_files[0])

print("PES scan molecule file:", sample_files[0])
print("PES scan molecule formula:", atoms0.get_chemical_formula())
print("Atomic symbols:", atoms0.get_chemical_symbols())
print("Positions:")
print(atoms0.get_positions())

scan = make_bond_scan(atoms0, calc)

if scan is not None:
    pair, r0, distances, energies = scan

    print("Scanned bond atom indices:", pair)
    print("Scanned bond elements:",
          atoms0[pair[0]].symbol, "-", atoms0[pair[1]].symbol)
    print("Equilibrium bond distance:", r0)

    plt.figure(figsize=(7, 5))
    plt.plot(distances, energies, marker="o", markersize=3)
    plt.axvline(r0, linestyle="--", linewidth=1)

    plt.xlabel("Stretched bond distance / Å")
    plt.ylabel("Predicted BPNN energy")
    plt.title(
        f"Predicted 1D PES scan: {os.path.basename(sample_files[0])}, "
        f"{atoms0.get_chemical_formula()}, "
        f"bond {pair[0]}-{pair[1]} "
        f"({atoms0[pair[0]].symbol}-{atoms0[pair[1]].symbol})"
    )
    plt.tight_layout()

    path = os.path.join(OUT_DIR, "predicted_1d_bond_pes_scan.png")
    plt.savefig(path, dpi=300)
    plt.close()

    print(f"Saved: {path}")


def plot_tensorboard_scalars(model_dir, out_dir):
    """
    Plot available TensorBoard scalar summaries from training.
    """

    try:
        from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    except Exception as exc:
        print("TensorBoard scalar plotting skipped.")
        print("Reason:", repr(exc))
        return

    event_files = []
    for root, _, files in os.walk(model_dir):
        for name in files:
            if name.startswith("events.out.tfevents"):
                event_files.append(os.path.join(root, name))

    if not event_files:
        print("No TensorBoard event files found.")
        return

    event_file = max(event_files, key=os.path.getmtime)

    ea = EventAccumulator(event_file)
    ea.Reload()

    scalar_tags = ea.Tags().get("scalars", [])

    if not scalar_tags:
        print("No scalar summaries found in TensorBoard event file.")
        return

    for tag in scalar_tags:
        events = ea.Scalars(tag)

        steps = np.array([e.step for e in events])
        values = np.array([e.value for e in events])

        safe_tag = tag.replace("/", "_").replace(" ", "_")

        plt.figure(figsize=(7, 5))
        plt.plot(steps, values)
        plt.xlabel("Training step")
        plt.ylabel(tag)
        plt.title(f"Training curve: {tag}")
        plt.tight_layout()

        path = os.path.join(out_dir, f"training_curve_{safe_tag}.png")
        plt.savefig(path, dpi=300)
        plt.close()

    print(f"Saved TensorBoard scalar plots to: {out_dir}")


plot_tensorboard_scalars(MODEL_DIR, OUT_DIR)

print("\nDone.")
print(f"All results saved in: {OUT_DIR}")

Found 133885 QM9 files.
INFO:tensorflow:Using default config.
INFO:tensorflow:Using config: {'_model_dir': './BPNN_QM9_U0', '_tf_random_seed': None, '_save_summary_steps': 100, '_save_checkpoints_steps': None, '_save_checkpoints_secs': 600, '_session_config': allow_soft_placement: true
graph_options {
  rewrite_options {
    meta_optimizer_iterations: ONE
  }
}
, '_keep_checkpoint_max': 5, '_keep_checkpoint_every_n_hours': 10000, '_log_step_count_steps': 100, '_train_distribute': None, '_device_fn': None, '_protocol': None, '_eval_distribute': None, '_experimental_distribute': None, '_experimental_max_worker_delay_secs': None, '_session_creation_timeout_secs': 7200, '_checkpoint_save_graph_def': True, '_service': None, '_cluster_spec': ClusterSpec({}), '_task_type': 'worker', '_task_id': 0, '_global_id_in_cluster': 0, '_master': '', '_evaluation_master': '', '_is_chief': True, '_num_ps_replicas': 0, '_num_worker_replicas': 1}
Instructions for updating:
Use output_signature instead
Inst